# Credit Risk Model — Exploratory Data Analysis

**Dataset**: Xente eCommerce transaction data (95,662 transactions, 3,632 customers)  
**Goal**: Understand the data, validate feature engineering assumptions, and justify the RFM-based proxy default label.

## Contents
1. Load & Inspect
2. Missing Values & Data Quality
3. Transaction Distribution
4. Time Series Analysis
5. Product & Channel Analysis
6. Fraud Analysis
7. RFM Feature Engineering & Clustering
8. Feature Correlation
9. Proxy Label Validation


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/raw/data.csv'
print('Libraries loaded')

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['TransactionStartTime'])
print(f'Shape: {df.shape}')
print(f'Date range: {df.TransactionStartTime.min()} → {df.TransactionStartTime.max()}')
print(f'Unique customers: {df.CustomerId.nunique()}')
print(f'Unique products: {df.ProductId.nunique()}')
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 2. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print('\nNo missing values!' if missing.sum() == 0 else f'Total missing: {missing.sum()}')

In [ ]:
# Check for duplicate transactions
dupes = df.duplicated(subset='TransactionId').sum()
print(f'Duplicate TransactionIds: {dupes}')

# Amount vs Value consistency
df['amount_value_match'] = df['Amount'].abs().round(2) == df['Value'].round(2)
print(f'Amount/Value mismatch: {(~df.amount_value_match).sum()} rows')

## 3. Transaction Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Amount distribution (debits only)
debits = df[df['Amount'] > 0]['Amount']
axes[0].hist(debits.clip(upper=debits.quantile(0.99)), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount (debits, 99th pct clip)')
axes[0].set_xlabel('Amount')

# Log-scale amount
axes[1].hist(np.log1p(debits), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Log(Amount + 1)')
axes[1].set_xlabel('log(Amount)')

# Transactions per customer
txn_per_customer = df.groupby('CustomerId').size()
axes[2].hist(txn_per_customer.clip(upper=txn_per_customer.quantile(0.99)), bins=40, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Transactions per Customer (99th pct clip)')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.show()

print(f'Median transaction amount: {debits.median():.2f}')
print(f'Mean transactions per customer: {txn_per_customer.mean():.1f}')
print(f'Max transactions per customer: {txn_per_customer.max()}')

## 4. Time Series Analysis

In [ ]:
df['date'] = df['TransactionStartTime'].dt.date
df['hour'] = df['TransactionStartTime'].dt.hour
df['dayofweek'] = df['TransactionStartTime'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Daily transaction volume
daily = df.groupby('date').size()
axes[0].plot(daily.index, daily.values, linewidth=0.8, color='steelblue')
axes[0].set_title('Daily Transaction Volume')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Transactions')
axes[0].tick_params(axis='x', rotation=45)

# Hourly distribution
hourly = df.groupby('hour').size()
axes[1].bar(hourly.index, hourly.values, color='coral', edgecolor='white')
axes[1].set_title('Transactions by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Transactions')

plt.tight_layout()
plt.show()

## 5. Product & Channel Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top product categories
cat_counts = df['ProductCategory'].value_counts().head(10)
axes[0].barh(cat_counts.index[::-1], cat_counts.values[::-1], color='steelblue')
axes[0].set_title('Top 10 Product Categories')
axes[0].set_xlabel('Transaction Count')

# Channel distribution
channel_counts = df['ChannelId'].value_counts()
axes[1].pie(channel_counts.values, labels=channel_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Transaction Channel Distribution')

plt.tight_layout()
plt.show()

## 6. Fraud Analysis

In [ ]:
fraud_rate = df['FraudResult'].mean()
print(f'Overall fraud rate: {fraud_rate:.4f} ({fraud_rate*100:.2f}%)')
print(f'Fraud transactions: {df.FraudResult.sum()} / {len(df)}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fraud rate by product category
fraud_by_cat = df.groupby('ProductCategory')['FraudResult'].mean().sort_values(ascending=False)
axes[0].barh(fraud_by_cat.index[::-1], fraud_by_cat.values[::-1], color='salmon')
axes[0].set_title('Fraud Rate by Product Category')
axes[0].set_xlabel('Fraud Rate')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# Fraud amount distribution vs normal
fraud_amounts = df[df['FraudResult'] == 1]['Amount'].abs()
normal_amounts = df[df['FraudResult'] == 0]['Amount'].abs()
axes[1].hist(np.log1p(normal_amounts.clip(upper=normal_amounts.quantile(0.99))), 
             bins=40, alpha=0.6, label='Normal', color='steelblue')
axes[1].hist(np.log1p(fraud_amounts.clip(upper=fraud_amounts.quantile(0.99))), 
             bins=40, alpha=0.6, label='Fraud', color='salmon')
axes[1].set_title('Log(Amount) — Fraud vs Normal')
axes[1].set_xlabel('log(Amount)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. RFM Feature Engineering & Clustering

Since the dataset has no ground-truth default label, we engineer a **proxy target variable** using RFM segmentation:
- **Recency**: Days since last transaction (higher = more dormant = riskier)
- **Frequency**: Number of transactions (lower = less engaged = riskier)  
- **Monetary**: Total spend (lower = less valuable = riskier)

K-Means clustering (k=3) groups customers, and the cluster with highest recency + lowest frequency/monetary is labelled as the proxy default group.

In [ ]:
from src.data_processing import compute_rfm, assign_proxy_label, compute_aggregate_features

rfm = compute_rfm(df)
rfm_labeled = assign_proxy_label(rfm)

print('RFM Summary:')
print(rfm_labeled.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(1))
print(f'\nProxy default rate: {rfm_labeled.is_default.mean():.2%}')
print(f'Default customers: {rfm_labeled.is_default.sum()} / {len(rfm_labeled)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {0: 'salmon', 1: 'steelblue', 2: 'mediumseagreen'}

for cluster in rfm_labeled['Cluster'].unique():
    subset = rfm_labeled[rfm_labeled['Cluster'] == cluster]
    label = f'Cluster {cluster} ({"HIGH RISK" if subset.is_default.iloc[0] == 1 else "low risk"})'
    axes[0].scatter(subset['Recency'], subset['Frequency'], 
                    alpha=0.4, s=10, label=label, color=colors.get(cluster, 'gray'))
    axes[1].scatter(subset['Recency'], np.log1p(subset['Monetary']), 
                    alpha=0.4, s=10, color=colors.get(cluster, 'gray'))
    axes[2].scatter(subset['Frequency'], np.log1p(subset['Monetary']), 
                    alpha=0.4, s=10, color=colors.get(cluster, 'gray'))

axes[0].set_xlabel('Recency (days)'); axes[0].set_ylabel('Frequency'); axes[0].set_title('Recency vs Frequency')
axes[1].set_xlabel('Recency (days)'); axes[1].set_ylabel('log(Monetary)'); axes[1].set_title('Recency vs Monetary')
axes[2].set_xlabel('Frequency'); axes[2].set_ylabel('log(Monetary)'); axes[2].set_title('Frequency vs Monetary')
axes[0].legend(fontsize=8)

plt.suptitle('RFM Cluster Visualization', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 8. Feature Correlation

In [ ]:
from src.data_processing import build_feature_matrix

features = build_feature_matrix(df)

corr_cols = ['Recency', 'Frequency', 'Monetary', 'AvgTransactionValue',
             'StdTransactionValue', 'FraudRate', 'UniqueProducts',
             'UniqueChannels', 'NightTxnRatio', 'is_default']

corr = features[corr_cols].corr()

plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', 
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations with target
print('\nCorrelation with is_default:')
print(corr['is_default'].drop('is_default').sort_values(key=abs, ascending=False).round(3))

## 9. Proxy Label Validation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
feature_list = ['Recency', 'Frequency', 'Monetary', 'AvgTransactionValue', 'FraudRate', 'NightTxnRatio']

for ax, feat in zip(axes.flat, feature_list):
    good = features[features['is_default'] == 0][feat]
    bad  = features[features['is_default'] == 1][feat]
    
    clip_val = features[feat].quantile(0.99)
    ax.hist(good.clip(upper=clip_val), bins=30, alpha=0.6, label='Good (0)', color='steelblue', density=True)
    ax.hist(bad.clip(upper=clip_val),  bins=30, alpha=0.6, label='Bad (1)',  color='salmon',    density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Good vs Bad (Proxy Label)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print('=== Proxy Label Summary ===')
print(features.groupby('is_default')[feature_list].mean().round(2).T)
print(f'\nClass balance: {features.is_default.value_counts().to_dict()}')
print('\nKey insight: High-risk customers (is_default=1) have higher Recency (more dormant),'
      ' lower Frequency, and lower Monetary value — consistent with credit risk intuition.')